# Day 3 — Graph Mechanics and Observability

Days 1 and 2 built agents. Day 3 opens the hood. We reuse the same knapsack bug and a multi-state routing graph, and look at *how it runs and how to observe it*:

1. **Checkpoints and state updates** — what a checkpointer stores, and how `update_state` edits it.
2. **Entering at a specific node** — how to start a graph from a given state at a chosen node instead of `START`.
3. **All run data in MLflow** — how every run of Simple Retry, Multi-State, and Human-in-the-Loop shows up as traces you can list and filter.

No new agent concept today — this is a mechanics-and-observability lab. (Parallel execution now lives in Day 2.)

## 0. Connect, enable tracing, and build the shared graph

`autolog()` records every graph run as an MLflow **trace**; all of today's runs go to one experiment so Part 3 can list them together. We reuse a multi-state routing graph (`triage` -> one specialist -> `finalize`) as the running example.

In [ ]:
from langchain_openai import ChatOpenAI
import mlflow

llm = ChatOpenAI(
    base_url="http://127.0.0.1:5001/gateway/mlflow/v1",
    api_key="not-needed",
    model="workshop-gemini",
)

mlflow.set_tracking_uri("http://127.0.0.1:5001")
EXPERIMENT = "day-3-graph-mechanics"
mlflow.set_experiment(EXPERIMENT)
mlflow.langchain.autolog()

In [ ]:
from typing import Annotated, Literal, TypedDict
from operator import add
from pydantic import BaseModel
from langgraph.graph import END, START, StateGraph
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
import sys
sys.path.append("../..")
from graph_image import show_graph

problem_statement = "Choose items within the capacity and maximize value. Each item may be selected at most once."

buggy_code = """
n, capacity = map(int, input().split())
items = [tuple(map(int, input().split())) for _ in range(n)]
dp = [0] * (capacity + 1)
for weight, value in items:
    for current in range(weight, capacity + 1):
        dp[current] = max(dp[current], dp[current - weight] + value)
print(dp[capacity])
"""

TRIAGE_PROMPT = "Classify the primary Python bug as syntax, runtime, or logic."
SPECIALIST_PROMPT = "You are a {category} bug specialist. Give the root cause in one sentence. Do not invent missing context."

class Triage(BaseModel):
    category: Literal["syntax", "runtime", "logic"]

class Diagnosis(BaseModel):
    category: str
    cause: str

triage_llm = llm.with_structured_output(Triage)
specialist_llm = llm.with_structured_output(Diagnosis)

class MultiState(TypedDict, total=False):
    problem: str
    code: str
    category: str
    answer: str
    notes: Annotated[list, add]   # reduced key: update_state appends here

def triage(state: MultiState):
    result = triage_llm.invoke([
        {"role": "system", "content": TRIAGE_PROMPT},
        {"role": "user", "content": f"Problem:\n{state['problem']}\n\nCode:\n{state['code']}"},
    ])
    return {"category": result.category}

def specialist_node(category: str):
    def node(state: MultiState):
        result = specialist_llm.invoke([
            {"role": "system", "content": SPECIALIST_PROMPT.format(category=category)},
            {"role": "user", "content": f"Problem:\n{state['problem']}\n\nCode:\n{state['code']}"},
        ])
        return {"answer": f"[{category}] {result.cause}"}
    return node

def finalize(state: MultiState):
    return {"notes": [f"resolved via {state['category']} specialist"]}

def route(state: MultiState) -> Literal["syntax", "runtime", "logic"]:
    return state["category"]

mb = StateGraph(MultiState)
mb.add_node("triage", triage)
mb.add_node("syntax", specialist_node("syntax"))
mb.add_node("runtime", specialist_node("runtime"))
mb.add_node("logic", specialist_node("logic"))
mb.add_node("finalize", finalize)
mb.add_edge(START, "triage")
mb.add_conditional_edges("triage", route, {"syntax": "syntax", "runtime": "runtime", "logic": "logic"})
for name in ["syntax", "runtime", "logic"]:
    mb.add_edge(name, "finalize")
mb.add_edge("finalize", END)

multi_graph = mb.compile()
show_graph(multi_graph)

# Part 1 — Checkpoints and state updates

Compile a graph with a **checkpointer** and every superstep writes a checkpoint: a snapshot of the whole state, plus which nodes run next and a pointer to its parent. The chain of snapshots is the thread's history.

`get_state(config)` returns the latest snapshot; `get_state_history(config)` returns them all, newest first; `update_state(config, values, as_node=...)` writes a *new* checkpoint as if `as_node` had produced `values` — reducers apply, so an update to a reduced key (`notes`) is merged, not replaced.

In [ ]:
mem_graph = mb.compile(checkpointer=MemorySaver())
config = {"configurable": {"thread_id": "mechanics-1"}}

mem_graph.invoke({"problem": problem_statement, "code": buggy_code}, config)

snap = mem_graph.get_state(config)
print("state keys :", list(snap.values.keys()))
print("next       :", snap.next)   # () means the run is complete
print("checkpoint :", snap.config["configurable"]["checkpoint_id"][:8])
print("source     :", snap.metadata.get("source"), "| step:", snap.metadata.get("step"))
parent = snap.parent_config["configurable"]["checkpoint_id"][:8] if snap.parent_config else None
print("parent     :", parent)

In [ ]:
history = list(mem_graph.get_state_history(config))
print(f"{len(history)} checkpoints, newest first:\n")
for s in history:
    step = str(s.metadata.get("step"))
    print(f"  step {step:>3}  next={s.next}  ckpt={s.config['configurable']['checkpoint_id'][:8]}")

In [ ]:
new_config = mem_graph.update_state(
    config,
    {"notes": ["[human] verified: iterate capacity downward"]},
    as_node="finalize",
)

updated = mem_graph.get_state(config)
print("new checkpoint :", new_config["configurable"]["checkpoint_id"][:8])
print("notes          :", updated.values["notes"])   # finalize's note + the manual note (reducer appended)

A snapshot (`StateSnapshot`) carries: `values` (the state), `next` (nodes still to run — `()` when finished), `config` (the `thread_id` plus this checkpoint's id), `metadata` (`source`, `step`), and `parent_config` (the previous checkpoint). `update_state` did not overwrite `notes`; because that key has an `add` reducer, the manual note was appended to `finalize`'s note, and a fresh checkpoint now sits on top of the history.

# Part 2 — Start at a particular node

A normal `invoke(state)` enters at `START`. To begin somewhere else — to replay from a point, resume, or test one node with a known input — seed a thread's state and continue:

- `update_state(config, values, as_node="X")` writes those values as if node `X` had just produced them. LangGraph then computes what runs next from `X`'s outgoing edges.
- `invoke(None, config)` continues the thread from there, running only the remaining nodes.

Below we seed `category="logic"` as if `triage` had already run, so the graph enters the **logic specialist directly** and never calls the triage model.

In [ ]:
cfg = {"configurable": {"thread_id": "enter-at-logic"}}
seeded_graph = mb.compile(checkpointer=MemorySaver())

# Seed state as if triage already ran and chose "logic" -- no triage model call.
seeded_graph.update_state(
    cfg,
    {"problem": problem_statement, "code": buggy_code, "category": "logic"},
    as_node="triage",
)

print("next node to run:", seeded_graph.get_state(cfg).next)   # ('logic',)

result = seeded_graph.invoke(None, cfg)   # continues straight into the logic specialist
print(result["answer"])

`get_state(cfg).next` showed `('logic',)` before we continued — proof the graph was positioned to enter the logic node directly. This is the general pattern for starting mid-graph: set the state at the node you want to resume from, then `invoke(None)`. (Inside a graph, a node redirects the same way by *returning* `Command(goto=...)`, which the Day 2 `human_review` node does.)

# Part 3 — See all run data in MLflow

Every run so far was traced. Now we run the three agents you build across the week — **Simple Retry** (Day 1), **Multi-State routing** (Day 1), and **Human-in-the-Loop** (Day 2) — tag each trace with its agent name, then list them all from MLflow.

Watch the counts: Simple Retry and Multi-State each produce **one** trace, but the HITL run produces **several** — one per `invoke()` and per `Command(resume=...)` — because each pause-and-resume is a separate request on the same `thread_id`.

In [ ]:
class Attempt(BaseModel):
    diagnosis: str
    corrected_code: str
    solved: bool

retry_llm = llm.with_structured_output(Attempt)

class RetryState(TypedDict):
    problem: str
    code: str
    attempt: int
    limit: int
    history: list

def analyze(state: RetryState):
    previous = [a.model_dump() for a in state["history"]]
    result = retry_llm.invoke([
        {"role": "system", "content": "Debug the code. Review previous attempts. Set solved to true only when the fix addresses every issue."},
        {"role": "user", "content": f"Problem:\n{state['problem']}\n\nCode:\n{state['code']}\n\nPrevious attempts:\n{previous}"},
    ])
    return {"attempt": state["attempt"] + 1, "history": state["history"] + [result]}

def retry_route(state: RetryState) -> Literal["analyze", "end"]:
    if state["attempt"] >= state["limit"]:
        return "end"
    if state["attempt"] >= 3 and state["history"][-1].solved:
        return "end"
    return "analyze"

rb = StateGraph(RetryState)
rb.add_node("analyze", analyze)
rb.add_edge(START, "analyze")
rb.add_conditional_edges("analyze", retry_route, {"analyze": "analyze", "end": END})
simple_graph = rb.compile()

In [ ]:
class Step(BaseModel):
    diagnosis: str
    fix: str
    continue_loop: bool

step_llm = llm.with_structured_output(Step)

class HLState(TypedDict):
    problem: str
    code: str
    iteration: int
    limit: int
    steps: list

def hl_analyze(state: HLState):
    step = step_llm.invoke([
        {"role": "system", "content": "Inspect one bug per turn. Give a diagnosis and a fix. Set continue_loop true only if another bug may remain."},
        {"role": "user", "content": f"Problem:\n{state['problem']}\n\nCode:\n{state['code']}\n\nPrevious steps:\n{state['steps']}"},
    ])
    return {"iteration": state["iteration"] + 1, "steps": state["steps"] + [step.model_dump()]}

def hl_review(state: HLState) -> Command[Literal["hl_analyze", "__end__"]]:
    decision = interrupt({"question": "retry or end?"})
    if decision == "retry" and state["iteration"] < state["limit"]:
        return Command(goto="hl_analyze")
    return Command(goto=END)

hb = StateGraph(HLState)
hb.add_node("hl_analyze", hl_analyze)
hb.add_node("hl_review", hl_review)
hb.add_edge(START, "hl_analyze")
hb.add_edge("hl_analyze", "hl_review")
hl_graph = hb.compile(checkpointer=MemorySaver())

In [ ]:
def tag_last(agent: str):
    trace_id = mlflow.get_last_active_trace_id()
    mlflow.set_trace_tag(trace_id, "agent", agent)
    return trace_id

# 1) Simple Retry (Day 1)
simple_graph.invoke({"problem": problem_statement, "code": buggy_code, "attempt": 0, "limit": 5, "history": []})
tag_last("simple-retry")

# 2) Multi-State routing (Day 1)
multi_graph.invoke({"problem": problem_statement, "code": buggy_code})
tag_last("multi-state")

# 3) Human-in-the-loop (Day 2) -- resumed programmatically: retry once, then end
hitl_cfg = {"configurable": {"thread_id": "day-3-hitl"}}
result = hl_graph.invoke({"problem": problem_statement, "code": buggy_code, "iteration": 0, "limit": 5, "steps": []}, hitl_cfg)
tag_last("hitl")
for decision in ["retry", "end"]:
    if "__interrupt__" not in result:
        break
    result = hl_graph.invoke(Command(resume=decision), hitl_cfg)
    tag_last("hitl")

print("Simple Retry, Multi-State, and HITL have all run and been tagged.")

In [ ]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT)
traces = mlflow.search_traces(experiment_ids=[experiment.experiment_id])
print(f"{len(traces)} traces logged to '{EXPERIMENT}'.\n")

hitl_only = mlflow.search_traces(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.agent = 'hitl'",
)
print(f"HITL alone produced {len(hitl_only)} traces (one per invoke / resume).")

traces

`mlflow.search_traces(...)` returns a pandas DataFrame — one row per trace — with the request, response, status, latency (`execution_time_ms`), tags, and the full span tree. Filter it with a `filter_string` such as `tags.agent = 'hitl'`, or open **Experiments -> day-3-graph-mechanics -> Traces** in the MLflow UI and filter there. A trace's status `OK` means the run finished, not that the answer is correct — read the spans to judge quality.

## Break, inspect, reflect

- Open the Multi-State trace and confirm `triage` and the chosen specialist ran in sequence; open a Simple Retry trace and confirm the `analyze` spans repeat.
- Re-run the HITL cell on a fresh `thread_id` and compare its trace count with its `get_state_history` checkpoint count — checkpoints are finer-grained, and they vanish when the kernel restarts while the traces remain in MLflow.

**Exit questions:**
1. `update_state` created a new checkpoint instead of editing the old one. Why is an append-only history safer than in-place edits?
2. A seeded thread's `get_state(...).next` told you which node would run. When is starting a graph mid-way useful?
3. A trace's status is `OK` but the fix is wrong. Which evidence in the trace would reveal that?